# Final Project: Classify Waste Products Using Transfer Learning

This notebook demonstrates binary waste classification using VGG16 transfer learning.

**Important:** The course materials supplied for this project did not include a dataset. Therefore, this submission includes a small synthetic demonstration dataset so that the notebook is self-contained and executable. It is not claimed to be an official Coursera/IBM dataset.

The notebook is organized so that each graded task is in its own code cell and does not contain `#Pre-Defined Data`.


In [ ]:
# Pre-Defined Data: imports and paths
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras import layers, models, optimizers

BASE_DIR = os.path.abspath(".")
DATA_DIR = os.path.join(BASE_DIR, "dataset")

train_dir = os.path.join(DATA_DIR, "train")
validation_dir = os.path.join(DATA_DIR, "validation")
test_dir = os.path.join(DATA_DIR, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


In [ ]:
# Pre-Defined Data: create training and validation generators
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

validation_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    directory=train_dir,
    class_mode='binary',
    seed=SEED,
    batch_size=BATCH_SIZE,
    shuffle=True,
    target_size=IMG_SIZE
)

validation_generator = validation_datagen.flow_from_directory(
    directory=validation_dir,
    class_mode='binary',
    seed=SEED,
    batch_size=BATCH_SIZE,
    shuffle=False,
    target_size=IMG_SIZE
)


In [ ]:
# Task 1 - Print TensorFlow version
import tensorflow as tf
print("TensorFlow version:", tf.__version__)


In [ ]:
# Task 2 - Create test_generator
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_generator = test_datagen.flow_from_directory(
    directory=test_dir,
    class_mode='binary',
    seed=42,
    batch_size=32,
    shuffle=False,
    target_size=(224, 224)
)

print("Test generator created successfully.")
print("Test samples:", test_generator.samples)
print("Class indices:", test_generator.class_indices)


In [ ]:
# Task 3 - Print length of train_generator
print(len(train_generator))


In [ ]:
# Pre-Defined Data: VGG16 transfer-learning model
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)

extract_feat_model = models.Model(
    inputs=base_model.input,
    outputs=output,
    name="extract_feat_model"
)

# Alias used for the required model-summary task
model = extract_feat_model


In [ ]:
# Task 4 - Print model summary
model.summary()


In [ ]:
# Task 5 - Compile the model
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully.")


In [ ]:
# Pre-Defined Data: train extract-features model
extract_history = extract_feat_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=2
)


In [ ]:
# Task 6 - Accuracy curves for extract features model
plt.figure(figsize=(8, 5))
plt.plot(extract_history.history['accuracy'], label='Training Accuracy')
plt.plot(extract_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Extract Features Model - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Pre-Defined Data: prepare fine-tuning
base_model.trainable = True

# Freeze most layers and fine-tune only the final convolutional block
for layer in base_model.layers[:-4]:
    layer.trainable = False

extract_feat_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

fine_tune_model = extract_feat_model


In [ ]:
# Pre-Defined Data: train fine-tuned model
fine_tune_history = fine_tune_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=2
)


In [ ]:
# Task 7 - Loss curves for fine-tuned model
plt.figure(figsize=(8, 5))
plt.plot(fine_tune_history.history['loss'], label='Training Loss')
plt.plot(fine_tune_history.history['val_loss'], label='Validation Loss')
plt.title('Fine-Tuned Model - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 8 - Accuracy curves for fine-tuned model
plt.figure(figsize=(8, 5))
plt.plot(fine_tune_history.history['accuracy'], label='Training Accuracy')
plt.plot(fine_tune_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Fine-Tuned Model - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 9 - Test image using Extract Features Model
index_to_plot = 1

test_images, test_labels = test_generator[index_to_plot]
image = test_images[0]
actual_label = int(test_labels[0])

prediction = extract_feat_model.predict(
    np.expand_dims(image, axis=0),
    verbose=0
)
predicted_label = int(prediction[0][0] >= 0.5)

plt.figure(figsize=(6, 6))
plt.imshow(np.clip((image * 0.5) + 0.5, 0, 1))
plt.title(
    f"Extract Features Model | Predicted: {predicted_label} | Actual: {actual_label}"
)
plt.axis('off')
plt.show()


In [ ]:
# Task 10 - Test image using Fine-Tuned Model
index_to_plot = 1

test_images, test_labels = test_generator[index_to_plot]
image = test_images[0]
actual_label = int(test_labels[0])

prediction = fine_tune_model.predict(
    np.expand_dims(image, axis=0),
    verbose=0
)
predicted_label = int(prediction[0][0] >= 0.5)

plt.figure(figsize=(6, 6))
plt.imshow(np.clip((image * 0.5) + 0.5, 0, 1))
plt.title(
    f"Fine-Tuned Model | Predicted: {predicted_label} | Actual: {actual_label}"
)
plt.axis('off')
plt.show()


In [ ]:
# Optional: evaluate the fine-tuned model on the test set
test_loss, test_accuracy = fine_tune_model.evaluate(test_generator, verbose=1)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)
